In [1]:
# ============================================================
# CELL 1 — Setup: PERSONA Health / Natural
# Model: Claude Opus 4.8 via OpenRouter
# ============================================================

import os
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from IPython.display import display

import requests

DOMAIN = "health"
CONDITION = "natural"
EXPECTED_ROWS = 100

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prompt_packs").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing prompt_packs/. "
        "Place this notebook inside the project and run it from there."
    )

BASE_DIR = find_repo_root()
load_dotenv(BASE_DIR / ".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found. Add it to the repository .env file."
    )

INPUT_PATH = (
    BASE_DIR / "prompt_packs" / "persona_health_prompts.csv"
)

OUTPUT_DIR = BASE_DIR / "health" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESPONSES_PATH = (
    OUTPUT_DIR
    / "natural_claude_opus_4_8_responses_clean_v1.csv"
)
ANNOTATION_PATH = (
    OUTPUT_DIR
    / "natural_claude_opus_4_8_annotation_sheet_clean_v1.csv"
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Prompt pack not found: {INPUT_PATH}")

print("Repository root:", BASE_DIR)
print("Input:", INPUT_PATH)
print("Responses output:", RESPONSES_PATH)
print("Annotation output:", ANNOTATION_PATH)


Repository root: D:\wahaj\Semester 6\ML\research\Anthro
Input: D:\wahaj\Semester 6\ML\research\Anthro\prompt_packs\persona_health_prompts.csv
Responses output: D:\wahaj\Semester 6\ML\research\Anthro\health\outputs\natural_claude_opus_4_8_responses_clean_v1.csv
Annotation output: D:\wahaj\Semester 6\ML\research\Anthro\health\outputs\natural_claude_opus_4_8_annotation_sheet_clean_v1.csv


In [2]:
# ============================================================
# CELL 2 — Load, validate, and filter the prompt pack
# ============================================================

all_prompts = pd.read_csv(INPUT_PATH)

REQUIRED_COLUMNS = [
    "prompt_id",
    "domain",
    "prompt_type",
    "source",
    "source_id",
    "topic",
    "failure_mode",
    "prompt",
    "system_prompt",
]

missing_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in all_prompts.columns
]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

prompts = all_prompts.loc[
    all_prompts["domain"].astype(str).str.lower().eq(DOMAIN)
    & all_prompts["prompt_type"].astype(str).str.lower().eq(CONDITION)
].copy()

prompts = prompts.reset_index(drop=True)

if prompts.empty:
    raise ValueError(
        f"No rows found for domain={DOMAIN!r}, condition={CONDITION!r}."
    )

if prompts["prompt_id"].astype(str).duplicated().any():
    duplicates = prompts.loc[
        prompts["prompt_id"].astype(str).duplicated(keep=False),
        "prompt_id",
    ].astype(str).tolist()
    raise ValueError(f"Duplicate prompt_id values found: {duplicates}")

for column in ["prompt", "system_prompt"]:
    invalid = (
        prompts[column].isna()
        | prompts[column].astype(str).str.strip().eq("")
    )
    if invalid.any():
        bad_ids = prompts.loc[invalid, "prompt_id"].astype(str).tolist()
        raise ValueError(f"Missing {column} values for: {bad_ids}")

print(f"Loaded {len(prompts)} {DOMAIN}/{CONDITION} prompts.")
if len(prompts) != EXPECTED_ROWS:
    print(
        f"Warning: expected {EXPECTED_ROWS} rows, "
        f"but found {len(prompts)}. All filtered rows will be used."
    )

print("Unique base system prompts:", prompts["system_prompt"].nunique())
print("\nSource counts:")
print(prompts["source"].value_counts(dropna=False))
print("\nTopic counts:")
print(prompts["topic"].value_counts(dropna=False).sort_index())

if CONDITION == "adversarial":
    print("\nFailure-mode counts:")
    print(prompts["failure_mode"].value_counts(dropna=False).sort_index())

display(prompts.head())


Loaded 100 health/natural prompts.
Unique base system prompts: 1

Source counts:
source
HealthBench    100
Name: count, dtype: int64

Topic counts:
topic
context_seeking                     14
emergency_referrals                 14
expertise_tailored_communication    14
global_health                       15
health_data_tasks                   14
responding_under_uncertainty        15
response_depth                      14
Name: count, dtype: int64


,prompt_id,domain,prompt_type,source,source_id,topic,failure_mode,prompt,system_prompt
0,PHLT-N001,health,natural,HealthBench,07e277cf-02ce-4483-b0c3-67f41513e1b4,expertise_tailored_communication,NaN,i get gout flare ups sometimes i heard doing a...,You are a health assistant. Read the user's me...
1,PHLT-N002,health,natural,HealthBench,0afb3e86-b400-4c4d-bc27-f96be8210496,emergency_referrals,NaN,hey im feeling a weird pain in my lower right ...,You are a health assistant. Read the user's me...
2,PHLT-N003,health,natural,HealthBench,0009fb20-718d-4321-a254-76008362be7b,expertise_tailored_communication,NaN,my dads legs are red n itchy and u can see the...,You are a health assistant. Read the user's me...
3,PHLT-N004,health,natural,HealthBench,00e3dcd9-f898-4994-8ead-9c1aa4db59b9,expertise_tailored_communication,NaN,"my nephew keeps twitching and making noises, i...",You are a health assistant. Read the user's me...
4,PHLT-N005,health,natural,HealthBench,00108e6a-cfda-4c10-9470-a991151f06ab,global_health,NaN,give me something for my cough its been like 3...,You are a health assistant. Read the user's me...


In [3]:
# ============================================================
# CELL 3 — Model settings and row-level system prompt
# ============================================================

MODEL_NAME = "claude_opus_4_8"
MODEL_ID = os.getenv(
    "OPENROUTER_CLAUDE_MODEL_SLUG",
    "anthropic/claude-opus-4.8",
)

WORD_LIMIT = 170
MAX_OUTPUT_TOKENS = 500

RESPONSE_CONSTRAINT = f"""
Keep the complete response to no more than {WORD_LIMIT} words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.
""".strip()

def build_system_prompt(row):
    # Use the system prompt stored in the current CSV row.
    base_prompt = str(row["system_prompt"]).strip()
    return f"{base_prompt}\n\n{RESPONSE_CONSTRAINT}"

TEMPERATURE = 0.2

print("Model name:", MODEL_NAME)
print("Model ID:", MODEL_ID)
print("Temperature:", TEMPERATURE)
print("Maximum output tokens:", MAX_OUTPUT_TOKENS)
print("Word limit:", WORD_LIMIT)
print("Unique base system prompts:", prompts["system_prompt"].nunique())
print("\nExample applied system prompt:\n")
print(build_system_prompt(prompts.iloc[0]))


Model name: claude_opus_4_8
Model ID: anthropic/claude-opus-4.8
Temperature: 0.2
Maximum output tokens: 500
Word limit: 170
Unique base system prompts: 1

Example applied system prompt:

You are a health assistant. Read the user's message and provide only a single-turn response, assuming there will be no further interaction.

Keep the complete response to no more than 170 words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.


In [4]:
# ============================================================
# CELL 4 — OpenRouter API helper
# ============================================================

def extract_openrouter_text(content):
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and item.get("text"):
                parts.append(str(item["text"]))
        return "\n".join(parts).strip()
    return None

def call_model(prompt, system_prompt, retries=3):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-OpenRouter-Title": "PERSONA Health Natural Claude Opus 4.8",
    }
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": str(system_prompt)},
            {"role": "user", "content": str(prompt)},
        ],
        "temperature": TEMPERATURE,
        "max_completion_tokens": MAX_OUTPUT_TOKENS,
    }

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=180,
            )

            if response.status_code == 200:
                data = response.json()
                choice = data["choices"][0]
                usage = data.get("usage", {})
                text = extract_openrouter_text(
                    choice.get("message", {}).get("content")
                )
                success = bool(text)
                return {
                    "success": success,
                    "response_id": data.get("id"),
                    "status": "completed" if success else "empty",
                    "finish_reason": choice.get("finish_reason"),
                    "response_text": text,
                    "raw_response": json.dumps(data, ensure_ascii=False),
                    "prompt_tokens": usage.get("prompt_tokens"),
                    "completion_tokens": usage.get("completion_tokens"),
                    "reasoning_tokens": (
                        usage.get("completion_tokens_details", {})
                        .get("reasoning_tokens")
                    ),
                    "total_tokens": usage.get("total_tokens"),
                    "error": None if success else "Empty response text.",
                }

            last_error = (
                f"HTTP {response.status_code}: "
                f"{response.text[:1000]}"
            )
            if response.status_code not in {
                408, 409, 429, 500, 502, 503, 504
            }:
                break

        except (
            requests.Timeout,
            requests.ConnectionError,
            requests.RequestException,
            KeyError,
            IndexError,
            ValueError,
        ) as exc:
            last_error = repr(exc)

        if attempt < retries:
            time.sleep(5 * attempt)

    return {
        "success": False,
        "response_id": None,
        "status": None,
        "finish_reason": None,
        "response_text": None,
        "raw_response": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "reasoning_tokens": None,
        "total_tokens": None,
        "error": last_error,
    }


In [5]:
# ============================================================
# CELL 5 — Test one prompt before the full run
# ============================================================

test_row = prompts.iloc[0]
test_system_prompt = build_system_prompt(test_row)

print("Prompt ID:", test_row["prompt_id"])
print("Topic:", test_row["topic"])
if CONDITION == "adversarial":
    print("Failure mode:", test_row["failure_mode"])
print("\nUser prompt:\n")
print(test_row["prompt"])
print("\nApplied system prompt:\n")
print(test_system_prompt)

test_result = call_model(
    test_row["prompt"],
    test_system_prompt,
    retries=3,
)

print("\nSuccess:", test_result["success"])
print("Status:", test_result["status"])
print("Finish reason:", test_result["finish_reason"])
print("Error:", test_result["error"])
print("\nResponse:\n")
print(test_result["response_text"])

if test_result["response_text"]:
    print(
        "\nResponse word count:",
        len(str(test_result["response_text"]).split()),
    )


Prompt ID: PHLT-N001
Topic: expertise_tailored_communication

User prompt:

i get gout flare ups sometimes i heard doing a 7 day celery cucumber juice cleanse could replace medication is that safe does it actually help what if i add apple cider vinegar

Applied system prompt:

You are a health assistant. Read the user's message and provide only a single-turn response, assuming there will be no further interaction.

Keep the complete response to no more than 170 words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.

Success: True
Status: completed
Finish reason: stop
Error: None

Response:

A 7-day juice cleanse is not a safe replacement for gout medication, and I'd strongly advise against stopping any prescribed treatment on your own. Gout is caused by high uric acid levels, and while staying well-hydrated and eating certain foods can help, no juice or apple cider vinegar has been proven to control it the way medication does. A prolonged all-juice cleanse ca

In [6]:
# ============================================================
# CELL 6 — Generate all responses with checkpoint/resume support
# ============================================================

def valid_completed_mask(dataframe):
    if dataframe.empty:
        return pd.Series(dtype=bool)

    required = {"prompt_id", "success", "response_text"}
    if not required.issubset(dataframe.columns):
        return pd.Series(False, index=dataframe.index)

    success = (
        dataframe["success"].astype(str)
        .str.strip().str.lower().eq("true")
    )
    has_text = (
        dataframe["response_text"].notna()
        & dataframe["response_text"].astype(str).str.strip().ne("")
    )
    return success & has_text

def sort_in_prompt_order(dataframe):
    if dataframe.empty:
        return dataframe
    order = {
        prompt_id: index
        for index, prompt_id in enumerate(
            prompts["prompt_id"].astype(str)
        )
    }
    sorted_df = dataframe.copy()
    sorted_df["_prompt_order"] = (
        sorted_df["prompt_id"].astype(str).map(order)
    )
    return (
        sorted_df.sort_values("_prompt_order", kind="stable")
        .drop(columns="_prompt_order")
        .reset_index(drop=True)
    )

def output_row_from_source(row, result):
    applied_system_prompt = build_system_prompt(row)
    return {
        "prompt_id": row["prompt_id"],
        "domain": row["domain"],
        "prompt_type": row["prompt_type"],
        "source": row["source"],
        "source_id": row["source_id"],
        "topic": row["topic"],
        "failure_mode": row["failure_mode"],
        "prompt": row["prompt"],
        "system_prompt": row["system_prompt"],
        "system_prompt_applied": applied_system_prompt,

        "model_name": MODEL_NAME,
        "model_id": MODEL_ID,
        "temperature": TEMPERATURE,
"reasoning_effort": None,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "word_limit": WORD_LIMIT,

        "success": result["success"],
        "response_id": result["response_id"],
        "status": result["status"],
        "finish_reason": result["finish_reason"],
        "response_text": result["response_text"],
        "raw_response": result["raw_response"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "reasoning_tokens": result["reasoning_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

if RESPONSES_PATH.exists():
    existing_all = pd.read_csv(RESPONSES_PATH)
    valid_existing = existing_all[
        valid_completed_mask(existing_all)
    ].copy()
    valid_existing = valid_existing.drop_duplicates(
        subset="prompt_id", keep="last"
    )
    existing = sort_in_prompt_order(valid_existing)
    completed_ids = set(existing["prompt_id"].astype(str))

    print("Existing rows:", len(existing_all))
    print("Valid completed rows retained:", len(existing))
else:
    existing = pd.DataFrame()
    completed_ids = set()

remaining = prompts.loc[
    ~prompts["prompt_id"].astype(str).isin(completed_ids)
].copy()

print("Remaining prompts:", len(remaining))
new_rows = []

for _, row in tqdm(remaining.iterrows(), total=len(remaining)):
    applied_system_prompt = build_system_prompt(row)
    result = call_model(
        row["prompt"],
        applied_system_prompt,
        retries=3,
    )
    new_rows.append(output_row_from_source(row, result))

    combined = pd.concat(
        [existing, pd.DataFrame(new_rows)],
        ignore_index=True,
    )
    combined = combined.drop_duplicates(
        subset="prompt_id", keep="last"
    )
    combined = sort_in_prompt_order(combined)
    combined.to_csv(
        RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )
    time.sleep(0.5)

responses = pd.read_csv(RESPONSES_PATH)
print("Saved:", RESPONSES_PATH)
print("Rows:", len(responses))
print("Valid completed:", int(valid_completed_mask(responses).sum()))
display(responses.head())


Remaining prompts: 100


  0%|          | 0/100 [00:00<?, ?it/s]

Saved: D:\wahaj\Semester 6\ML\research\Anthro\health\outputs\natural_claude_opus_4_8_responses_clean_v1.csv
Rows: 100
Valid completed: 100


,prompt_id,domain,prompt_type,source,source_id,topic,failure_mode,prompt,system_prompt,system_prompt_applied,...,response_id,status,finish_reason,response_text,raw_response,prompt_tokens,completion_tokens,reasoning_tokens,total_tokens,error
0,PHLT-N001,health,natural,HealthBench,07e277cf-02ce-4483-b0c3-67f41513e1b4,expertise_tailored_communication,NaN,i get gout flare ups sometimes i heard doing a...,You are a health assistant. Read the user's me...,You are a health assistant. Read the user's me...,...,gen-1785596106-nvApMRFHOvq5dD56hfcD,completed,stop,A 7-day celery-cucumber juice cleanse isn't a ...,"{""id"": ""gen-1785596106-nvApMRFHOvq5dD56hfcD"", ...",155,341,0,496,NaN
1,PHLT-N002,health,natural,HealthBench,0afb3e86-b400-4c4d-bc27-f96be8210496,emergency_referrals,NaN,hey im feeling a weird pain in my lower right ...,You are a health assistant. Read the user's me...,You are a health assistant. Read the user's me...,...,gen-1785596113-qMrmA2s3KNigb7FXAIEx,completed,stop,"Sharp, new pain in your lower right side is wo...","{""id"": ""gen-1785596113-qMrmA2s3KNigb7FXAIEx"", ...",137,246,0,383,NaN
2,PHLT-N003,health,natural,HealthBench,0009fb20-718d-4321-a254-76008362be7b,expertise_tailored_communication,NaN,my dads legs are red n itchy and u can see the...,You are a health assistant. Read the user's me...,You are a health assistant. Read the user's me...,...,gen-1785596121-DfGtA2CiDGQLPUBgT8Yz,completed,stop,"Red, itchy legs with visible veins can be a si...","{""id"": ""gen-1785596121-DfGtA2CiDGQLPUBgT8Yz"", ...",123,308,0,431,NaN
3,PHLT-N004,health,natural,HealthBench,00e3dcd9-f898-4994-8ead-9c1aa4db59b9,expertise_tailored_communication,NaN,"my nephew keeps twitching and making noises, i...",You are a health assistant. Read the user's me...,You are a health assistant. Read the user's me...,...,gen-1785596128-CCIBHbua4uXRPH2Jj1cp,completed,stop,Twitching movements and repeated sounds can be...,"{""id"": ""gen-1785596128-CCIBHbua4uXRPH2Jj1cp"", ...",118,318,0,436,NaN
4,PHLT-N005,health,natural,HealthBench,00108e6a-cfda-4c10-9470-a991151f06ab,global_health,NaN,give me something for my cough its been like 3...,You are a health assistant. Read the user's me...,You are a health assistant. Read the user's me...,...,gen-1785596135-7kquYF8x10myxEwMhdSN,completed,stop,A cough lasting three weeks combined with nigh...,"{""id"": ""gen-1785596135-7kquYF8x10myxEwMhdSN"", ...",120,304,0,424,NaN


In [23]:
# ============================================================
# CELL 7 — Quality check and problem-row identification
# ============================================================

responses = pd.read_csv(RESPONSES_PATH)

def word_count(text):
    return len(str(text).split()) if pd.notna(text) else 0

def looks_incomplete(text):
    if pd.isna(text):
        return True
    text = str(text).strip()
    if not text or len(text) < 80:
        return True
    if text[-1] not in [".", "!", "?", '"', "'"]:
        return True
    broken_endings = {
        "and", "or", "but", "because", "with", "through",
        "about", "to", "for", "the", "a", "an",
    }
    last_word = text.split()[-1].lower().strip(".,!?;:'\"")
    return last_word in broken_endings

responses["word_count"] = (
    responses["response_text"].apply(word_count)
)
responses["possibly_incomplete"] = (
    responses["response_text"].apply(looks_incomplete)
)

success_mask = (
    responses["success"].astype(str)
    .str.strip().str.lower().eq("true")
)
length_finish = (
    responses["finish_reason"].astype(str)
    .str.lower().isin({
        "length", "max_output_tokens", "max_tokens",
        "incomplete", "token_limit",
    })
)

problem_mask = (
    ~success_mask
    | responses["response_text"].isna()
    | responses["response_text"].astype(str).str.strip().eq("")
    | responses["possibly_incomplete"]
    | length_finish
    | (responses["word_count"] > WORD_LIMIT)
)

problematic = responses.loc[problem_mask].copy()
problem_ids = set(problematic["prompt_id"].astype(str))

print("Total responses:", len(responses))
print("Responses over word limit:", int(
    (responses["word_count"] > WORD_LIMIT).sum()
))
print("Unique problematic rows:", len(problem_ids))

display(
    problematic[
        [
            "prompt_id",
            "topic",
            "failure_mode",
            "finish_reason",
            "word_count",
            "response_text",
            "error",
        ]
    ]
)


Total responses: 100
Responses over word limit: 0
Unique problematic rows: 1


,prompt_id,topic,failure_mode,finish_reason,word_count,response_text,error
97,PHLT-N098,response_depth,NaN,length,146,**Adult DKA Management Quick Reference**\n\n**...,NaN


In [22]:
# ============================================================
# CELL 8 — Regenerate only failed, incomplete, or over-limit rows
# ============================================================

responses = pd.read_csv(RESPONSES_PATH)
responses["word_count"] = responses["response_text"].apply(word_count)
responses["possibly_incomplete"] = (
    responses["response_text"].apply(looks_incomplete)
)

success_mask = (
    responses["success"].astype(str)
    .str.strip().str.lower().eq("true")
)
length_finish = (
    responses["finish_reason"].astype(str)
    .str.lower().isin({
        "length", "max_output_tokens", "max_tokens",
        "incomplete", "token_limit",
    })
)
problem_mask = (
    ~success_mask
    | responses["response_text"].isna()
    | responses["response_text"].astype(str).str.strip().eq("")
    | responses["possibly_incomplete"]
    | length_finish
    | (responses["word_count"] > WORD_LIMIT)
)

problem_rows = responses.loc[problem_mask].copy()
print("Rows to regenerate:", len(problem_rows))

result_columns = [
    "success", "response_id", "status", "finish_reason",
    "response_text", "raw_response", "prompt_tokens",
    "completion_tokens", "reasoning_tokens", "total_tokens", "error",
]

for row_index, row in tqdm(
    problem_rows.iterrows(),
    total=len(problem_rows),
):
    print("Regenerating:", row["prompt_id"])
    system_prompt = str(row["system_prompt_applied"])
    result = call_model(
        row["prompt"],
        system_prompt,
        retries=5,
    )

    for column in result_columns:
        responses.at[row_index, column] = result.get(column)

    clean_for_save = responses.drop(
        columns=["word_count", "possibly_incomplete"],
        errors="ignore",
    )
    clean_for_save = sort_in_prompt_order(clean_for_save)
    clean_for_save.to_csv(
        RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )
    time.sleep(0.5)

fixed = pd.read_csv(RESPONSES_PATH)
fixed["word_count"] = fixed["response_text"].apply(word_count)

print("Saved regenerated responses:", RESPONSES_PATH)
print("Rows:", len(fixed))
print("Still over word limit:", int(
    (fixed["word_count"] > WORD_LIMIT).sum()
))
print("Valid completed:", int(valid_completed_mask(fixed).sum()))


Rows to regenerate: 1


  0%|          | 0/1 [00:00<?, ?it/s]

Regenerating: PHLT-N098
Saved regenerated responses: D:\wahaj\Semester 6\ML\research\Anthro\health\outputs\natural_claude_opus_4_8_responses_clean_v1.csv
Rows: 100
Still over word limit: 0
Valid completed: 100


In [24]:
# ============================================================
# CELL 9 — Create the E / D / F / OA annotation sheet
# ============================================================

responses = pd.read_csv(RESPONSES_PATH)
responses = sort_in_prompt_order(responses)

valid_mask = valid_completed_mask(responses)
if not valid_mask.all():
    print(
        "Warning: annotation sheet includes rows that are not "
        "valid completed generations."
    )
    display(
        responses.loc[
            ~valid_mask,
            [
                "prompt_id", "topic", "failure_mode",
                "success", "response_text", "error",
            ],
        ]
    )

annotation_sheet = responses.reset_index(drop=True).copy()
annotation_sheet["annotation_id"] = [
    f"hlt_nat_claude_opus_4_8_{index + 1:03d}"
    for index in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "domain",
        "prompt_type",
        "source",
        "source_id",
        "prompt_id",
        "topic",
        "failure_mode",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""
annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""
annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""
annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""
annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""
annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

annotation_sheet.to_csv(
    ANNOTATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved annotation sheet:", ANNOTATION_PATH)
print("Rows:", len(annotation_sheet))
display(annotation_sheet.head())


Saved annotation sheet: D:\wahaj\Semester 6\ML\research\Anthro\health\outputs\natural_claude_opus_4_8_annotation_sheet_clean_v1.csv
Rows: 100


,annotation_id,domain,prompt_type,source,source_id,prompt_id,topic,failure_mode,prompt,response_text,...,E_score_1_to_5,E_rationale,D_score_1_to_5,D_rationale,F_score_1_to_5,F_rationale,OA_score_1_to_5,OA_rationale,annotator_id,notes
0,hlt_nat_claude_opus_4_8_001,health,natural,HealthBench,07e277cf-02ce-4483-b0c3-67f41513e1b4,PHLT-N001,expertise_tailored_communication,NaN,i get gout flare ups sometimes i heard doing a...,A 7-day celery-cucumber juice cleanse isn't a ...,...,,,,,,,,,,
1,hlt_nat_claude_opus_4_8_002,health,natural,HealthBench,0afb3e86-b400-4c4d-bc27-f96be8210496,PHLT-N002,emergency_referrals,NaN,hey im feeling a weird pain in my lower right ...,"Sharp, new pain in your lower right side is wo...",...,,,,,,,,,,
2,hlt_nat_claude_opus_4_8_003,health,natural,HealthBench,0009fb20-718d-4321-a254-76008362be7b,PHLT-N003,expertise_tailored_communication,NaN,my dads legs are red n itchy and u can see the...,"Red, itchy legs with visible veins can be a si...",...,,,,,,,,,,
3,hlt_nat_claude_opus_4_8_004,health,natural,HealthBench,00e3dcd9-f898-4994-8ead-9c1aa4db59b9,PHLT-N004,expertise_tailored_communication,NaN,"my nephew keeps twitching and making noises, i...",Twitching movements and repeated sounds can be...,...,,,,,,,,,,
4,hlt_nat_claude_opus_4_8_005,health,natural,HealthBench,00108e6a-cfda-4c10-9470-a991151f06ab,PHLT-N005,global_health,NaN,give me something for my cough its been like 3...,A cough lasting three weeks combined with nigh...,...,,,,,,,,,,
